# SpatialFusion: Xenium Ovarian 5k Dataset

adapted from: https://github.com/uhlerlab/spatialfusion/blob/main/tutorials/embed-and-finetune-sample.ipynb

uni.ipynb - generating embedding for .tif file
scgpt.ipynb - generating embedding for .h5ad file

## II. & III. Multimodal and Spatial encoding
With unimodal embeddings computed for H&E image patches and the ST profile, we next apply the SpatialFusion package to integrate these modalities and incorporate spatial context.

SpatialFusion proceeds in two stages:

1. **Multimodal encoding** with an AutoEncoder (AE): Align H&E and ST embeddings into a shared latent space to produce a joint multimodal embedding.
2. **Spatial encoding** with a graph convolutional network (GCN): Refine the multimodal embeddings by incorporating spatial neighborhood relationships between cells.
The pretrained checkpoints for these models are located in the data/checkpoint_* folders in the GitHub repository.

Note: Unimodal embeddings have been saved in the output_dir path, you can start from this step if prior steps have already been run.

In [1]:
# helper functions
import colorsys
from typing import Optional, Sequence, Mapping, Tuple, Dict, List

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb, to_hex
from sklearn.decomposition import PCA

def plot_embeddings(
    z1, z2, z_joint, labels, samples, label_title,
    palette="tab10", seed=42, max_cells=500_000
):

    rng = np.random.default_rng(seed)

    df = pd.DataFrame({"sample": samples, "label": labels}, index=z1.index)
    df["idx"] = np.arange(len(df))
    grouped = df.groupby("sample")

    # Determine how many cells per sample to keep
    total_samples = len(grouped)
    max_per_sample = max_cells // max(1, total_samples)

    # Subsample per sample group
    selected_indices = []
    for _, group in grouped:
        n = min(len(group), max_per_sample)
        selected_indices.extend(rng.choice(group["idx"].values, size=n, replace=False))

    # Subset everything
    z1 = z1.iloc[selected_indices]
    z2 = z2.iloc[selected_indices]
    z_joint = z_joint.iloc[selected_indices]
    labels = np.array(labels)[selected_indices]
    samples = np.array(samples)[selected_indices]

    # Apply PCA
    pca = PCA(n_components=2)
    z1_pca = pca.fit_transform(z1.values)
    z2_pca = pca.fit_transform(z2.values)
    z_joint_pca = pca.fit_transform(z_joint.values)

    # Shuffle for plot order
    #perm = rng.permutation(len(labels))
    #z1_pca = z1_pca[perm]
    #z2_pca = z2_pca[perm]
    #z_joint_pca = z_joint_pca[perm]
    #labels = labels[perm]

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    point_size = 5  # keep a single source of truth for both scatter and legend markers

    def _strip_spines(ax):
        # Remove ALL spines (no plot borders)
        for spine in ax.spines.values():
            spine.set_visible(False)
        # No ticks and equal aspect
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal")

    def plot_scatter(ax, emb, title, show_legend=False):
        sns.scatterplot(
            x=emb[:, 0],
            y=emb[:, 1],
            hue=labels,
            palette=palette,
            s=point_size,
            ax=ax,
            linewidth=0,
            alpha=0.8,
            legend="full" if show_legend else False,
        )
        ax.set_title(title, fontsize=25)
        _strip_spines(ax)

    plot_scatter(axes[0], z1_pca, "H&E (PCA)")
    plot_scatter(axes[1], z2_pca, "RNA (PCA)")
    plot_scatter(axes[2], z_joint_pca, "Joint (PCA)", show_legend=True)

    # Extract and remove subplot legend
    handles, labels_ = axes[2].get_legend_handles_labels()
    if axes[2].legend_ is not None:
        axes[2].legend_.remove()

    # Make legend marker sizes match the scatter point size
    # (handles from seaborn are PathCollections for the color items)
    for h in handles:
        if hasattr(h, "set_sizes"):
            h.set_sizes([point_size])  # one marker per legend entry

    # Add shared figure legend (text size follows rcParams; markers already matched)
    fig.legend(
        handles,
        labels_,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        title=label_title,
        scatterpoints=1,     # one marker per legend entry
        markerscale=10,     # keep scale = 1 to respect set_sizes above
        frameon=False,       # optional: no border around the legend
        fontsize=16,        # control legend text size here
        title_fontsize=18,
    )

    plt.tight_layout()
    plt.show()

In [5]:
from spatialfusion.embed.embed import AEInputs, run_full_embedding

# Standard library
import logging
import os
import warnings
import pathlib as pl

# Third-party libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import shapely.wkb
import tifffile
import timm
import torch
from PIL import Image
from tqdm.notebook import tqdm
from torchvision import transforms



[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
# previously computed embeddings
output_dir = '/Users/marissaesteban/Documents/experiments/spatialfusion/xenium_ovca_tutorial/'
uni_df = pd.read_parquet(pl.Path(output_dir) / 'UNI.parquet')
scgpt_df = pd.read_parquet(pl.Path(output_dir) / 'scGPT.parquet')

adata_path = "/Users/marissaesteban/Documents/data/xenium_ovca/data/tutadata_subset.h5ad"
adata = sc.read_h5ad(adata_path)

sample_name = "OVCA_Xenium"

In [13]:
# can run this for several samples at a time
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_uni=uni_df, z_scgpt=scgpt_df),
}

# this uses the average version
emb_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    device="cpu",
    combine_mode="average",
    spatial_key='spatial',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
    ae_batch_size=None, # optional, None automatically determines batch size based on input size and available memory 
    gcn_batch_size=None # optional, None uses full-graph GCN inference
)

[run_full_embedding] Using packaged pretrained AE checkpoint: /opt/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/spatialfusion/data/checkpoint_dir_ae/spatialfusion-multimodal-ae.pt
[run_full_embedding] Using packaged pretrained GCN checkpoint: /opt/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/spatialfusion/data/checkpoint_dir_gcn/spatialfusion-full-gcn.pt


Running GCN inference: 100%|██████████| 1/1 [00:00<00:00, 16.90it/s]


In [14]:
emb_df.head(10)

,0,1,2,3,4,5,6,7,8,9,sample_id,cell_id,celltype,X_coord,Y_coord
0,0.110017,0.101813,-0.247728,0.009807,-0.185939,-0.186653,-0.106890,0.163789,0.123904,0.301038,OVCA_Xenium,bodaggdc-1,Lymphocytes,8909.002585,8516.946833
1,0.106450,0.073020,-0.240751,-0.008678,-0.177073,-0.152422,-0.121217,0.158536,0.155930,0.279608,OVCA_Xenium,bodahgja-1,Lymphocytes,8839.379676,8573.668971
2,0.110974,0.090424,-0.244818,0.004175,-0.184774,-0.174996,-0.112353,0.163546,0.134517,0.293644,OVCA_Xenium,bodalbfd-1,Myeloid,8899.896676,8549.438262
3,0.044133,-0.079863,-0.239471,-0.065424,-0.085869,0.119841,-0.204218,0.098510,0.288894,0.123902,OVCA_Xenium,bodamehc-1,Myeloid,8062.427850,8699.193586
4,0.028509,-0.100555,-0.220043,-0.079985,-0.078658,0.129408,-0.198455,0.101788,0.278187,0.136972,OVCA_Xenium,bodanfem-1,Malignant,7948.017765,8781.114576
5,0.050539,-0.054368,-0.227621,-0.062868,-0.096222,0.085477,-0.198484,0.104352,0.290654,0.111739,OVCA_Xenium,bodangii-1,Malignant,8090.523823,8734.000379
6,0.041310,-0.091730,-0.229758,-0.065104,-0.084958,0.129151,-0.200120,0.098992,0.283135,0.117473,OVCA_Xenium,bodaonoh-1,Malignant,8044.218697,8709.041119
7,0.047254,-0.068795,-0.237164,-0.064207,-0.089132,0.104946,-0.199029,0.100582,0.286775,0.120748,OVCA_Xenium,bodapgnj-1,Malignant,8096.092915,8698.008941
8,0.034968,-0.104692,-0.242168,-0.057803,-0.076444,0.151438,-0.202073,0.094203,0.281512,0.115754,OVCA_Xenium,bodaplcb-1,Malignant,8041.488434,8658.176944
9,0.017660,-0.154827,-0.184396,-0.034125,-0.076466,0.165543,-0.160620,0.121874,0.225755,0.059604,OVCA_Xenium,bodbdiel-1,Malignant,7884.713835,8622.031773


## Multimodal encoding

In [15]:
from spatialfusion.utils.pkg_ckpt import resolve_pkg_ckpt
from spatialfusion.embed.embed import load_paired_ae, ae_from_arrays